# This notebook tries out the model + simulation framework from Bayesion_optimization_functions on simple ODEs.

In [6]:
from Bayesian_optimization_functions import *

In [7]:
# Create models

bc_cartesian = {("phi", "x"): ("dirichlet", "dirichlet"),
      ("phi", "y"): ("dirichlet", "dirichlet"),
      ("phi", "z"): ("dirichlet", "dirichlet")}

model_cartesian = Model(["dxx(phi) + dyy(phi) + dzz(phi) + rho"],
            ["phi(x, y, z)"],
            parameters=["rho(x,y,z)"],
            boundary_conditions=bc_cartesian)

bc_spherical = {("phi", "r"): ("dr(phi)", "dirichlet"),}

model_spherical = Model(["drr(phi) + 2 / r * dr(phi) + rho"],
            ["phi(r)"],
            parameters=["rho(r)"],
            boundary_conditions=bc_spherical)

In [8]:
# Initialize parameters
Q = 1
sigma = 1
a = 1
A = 100
V = 4/3 * np.pi * (A**3 - a**3) #Volume of the spherical shell
rho_0 = Q / V # uniform density of charge

# Cartesian space
N = 128
x = y = z = np.linspace(-A, A, N)
xx, yy, zz = np.meshgrid(x, y, z, indexing="ij")
rho_uniform = np.zeros_like(xx)
shell = np.where(xx**2 + yy**2 + zz**2>a**2)
rho_uniform[shell] = rho_0
phi = np.zeros_like(xx)
initial_fields_cartesian = model_cartesian.Fields(x=x, y=y, z=z, phi=phi,
                            rho = rho_uniform)

# Spherical (1D) space

# Uniform density of charge on spherical shell
r = np.linspace(a, A, N)
rho_uniform = np.zeros_like(r)
phi = np.zeros_like(r)
shell = np.where(r**2>a)
rho_uniform = rho_0
initial_fields_spherical = model_spherical.Fields(r=r, phi=phi,
                            rho = rho_uniform)

# Gaussian density of charge on sphere
r = np.linspace(0, A, N)
rho_gaussian = gaussian_charge(r, Q, sigma)
phi = np.zeros_like(r)
initial_fields_spherical = model_spherical.Fields(r=r, phi=phi,
                            rho = rho_gaussian)

In [11]:
# Run simulations

# simulation_cartesian = Simulation(model_cartesian, initial_fields_cartesian, dt=.1, tmax=1)
# container_cartesian = simulation_cartesian.attach_container()
# tmax, final_fields = simulation_cartesian.run()
# phi_sol_cartesian = container_cartesian.data.phi

simulation_spherical = Simulation(model_spherical, initial_fields_spherical, dt=1, tmax=100)
container_spherical = simulation_spherical.attach_container()
tmax, final_fields = simulation_spherical.run()
phi_sol_spherical = container_spherical.data.phi

a9524b running: t: 100: 100%|██████████| 100/100 [00:03<00:00, 26.86it/s]


In [ ]:
# Plot solution
import holoviews as hv
hv.notebook_extension("bokeh")

# hv.Dataset(phi_sol[:,:,:,N//2]).to(hv.Image, ["x", "y"])

In [12]:
# Uniform density of charge on spherical shell
# r = np.linspace(a, A, N)
# phi_analytical = analytical_potential(r, Q, sigma = 0)

# fig = go.Figure()
# fig.add_scatter(x = r, y = phi_analytical, mode = "markers + lines", name = "analytical potential")
# fig.add_scatter(x = r, y = phi_sol_spherical[-1, :], mode = "markers + lines", name = "potential integrated with finite difference")
# fig.show()

# Uniform density of charge on spherical shell
r = np.linspace(0, A, N)
phi_analytical = analytical_potential(r, Q, sigma)
phi_0 = Q / ( 4*np.pi ) * np.sqrt( 2 / (np.pi * sigma**2) )

fig = go.Figure()
fig.add_scatter(x = r, y = phi_analytical, mode = "markers + lines", name = "analytical potential")
fig.add_scatter(x = [0], y = [phi_0])
fig.add_scatter(x = r, y = phi_sol_spherical[-1, :] + phi_analytical[-1], mode = "markers + lines", name = "potential integrated with finite difference")
fig.show()

c:\Users\Luc\Documents\MEGAsync\PhD\RheoFlag\Code\Model\Bayesian_optimization_functions.py:54: RuntimeWarning:

divide by zero encountered in divide

c:\Users\Luc\Documents\MEGAsync\PhD\RheoFlag\Code\Model\Bayesian_optimization_functions.py:54: RuntimeWarning:

invalid value encountered in multiply

